# Problem set: Reinforcement Learning
## Introduction to AI 2
Emmanuel Keuleers

In this problem set, you will be asked to provide answers using python code and plain text.
If you struggle with the problems, you can submit questions in the discussion board.

Because this is a large notebook in which a lot of code depends on previous code, the code solutions are already given in an accompanying notebook.

You should try to make an effort to complete the code yourself. If it doesn't work, you can copy the solutions.   

Run the cell below to import the necessary libraries 

In [37]:
import random
import copy
from itertools import chain, cycle
from collections import defaultdict

In this assignment, you will implement a basic reinforcement learning algorithm to train an agent to navigate a simple grid world environment.

# 1. Creating the grid world

We will start by creating a grid world. Since the goal of our exercise is to let an agent navigate this world, we should place some obstacles and rewards in this world.

## 1.1 Create a grid

The first function you will write should just create the grid and add the goal state for the agent.

In [38]:
def create_grid(nrows, ncols):
    """Create a rectangular grid as a list of lists.
    Add the starting state in the first cell, represented by the symbol `S`.
    Add the goal state in the last cell, represented by the symbol `G`.
    These will be the top left and bottom right corner respectively when we visualize the grid.
    All other cells should be empty, represented by the symbol `.`.
    
    Arguments:
        nrows (int): The number of rows in the grid.
        ncols (int): The number of columns in the grid.
        
    Returns
        list: The grid as a list of lists with *nrows* rows and *ncols* columns.
    """
    # create the grid
    grid = [['.' for col in range(ncols)] for row in range(nrows)]
    # add the starting state
    grid[0][0] = 'S'
    # add the goal state
    grid[nrows-1][ncols-1] = 'G'
    # return the result
    return(grid)

In [41]:
# Test your code
# All tests should result in `True` if your code works correctly
print(create_grid(3, 4)==[['S', '.', '.', '.'], ['.', '.', '.', '.'], ['.', '.', '.', 'G']])
print(create_grid(2, 2)==[['S', '.'], ['.', 'G']])

True
True


## 1.2 Visualizing the grid
Next, we need a function to visualize the grid in 2 dimensions.

In [42]:
def visualize_grid(grid, agent_position = None):
    """
    Visualizes a grid world environment. 

    Arguments:
    - grid (list): A 2D list representing the grid world.
    - agent_position: The position of the agent in the grid. 
      If a position is given, the agent will be displayed as an "A" in the corresponding cell.

    Prints the grid world with each cell in a row separated by a space.
    """
    # copy the grid to avoid modifying the original
    vgrid = [row.copy() for row in grid]
    # add the agent position to the grid if given
    if agent_position:
        x,y=agent_position
        vgrid[x][y]="A"
    # print the grid
    for row in vgrid:
        print(" ".join(row))

In [45]:
# If all has gone well, so far, the code below should work and print the following grid:
# S . . .
# . A . .
# . . . .
# . . . G
test_grid = create_grid(5, 5)
visualize_grid(test_grid, agent_position = (1, 3))

S . . . .
. . . A .
. . . . .
. . . . .
. . . . G


## 1.2 Add objects
Next, we will add objects (obstacles or rewards) to the grid.

We will split the task in two parts. 

First, we will write a function to find the available positions in the grid. 

In [46]:
def find_available_positions(grid):
    """Find the positions of empty cells (cells containing a `.`) in a grid world.
    
    Arguments:
    - grid (list): A 2D list representing the grid world.

    Returns:
    - list: A list of tuples representing the available positions in the grid.
    """
    # Find the positions of all of the empty cells in the grid.
    available_positions = []
    for i in range(len(grid)):
        for j in range(len(grid[i])):
            if grid[i][j] == '.':
                available_positions.append((i,j))
    return(available_positions)

In [ ]:
# Test your code
# All tests should result in `True` if your code works correctly
test_grid = [['S', '#', '.', '#'], ['#', '.', '#', '.'], ['.', '.', '#', 'G']]
print(find_available_positions(test_grid)==[(0, 2), (1, 1), (1, 3), (2, 0), (2, 1)])

[(0, 2), (1, 1), (1, 3), (2, 0), (2, 1)]

Second, we will write a function to place desired objects randomly at available positions in the grid. 

This function should make use of `find_available_positions`.

In [49]:
def add_objects(grid, object_type = None, n = 0):
    """Add objects of a certain type to a grid at random positions.

    Objects can only be added to the grid at positions occupied by an empty cell.
    Obstacle objects should be represented by the symbol `X`. 
    Reward objects are represented by the symbol `R`.
    The function verifies if the grid has enough empty cells (marked by a `.`) to place the objects.

    Arguments:
        grid (list): The grid to which obstacles are added.
        object_type (str): The type of object to add to the grid. Can be either `obstacle` or `reward`.
        n (int): The number of objects of the specified type to add to the grid.
        
    Returns:
        `True` if the objects were added successfully, `False` otherwise.
    """
    # Set up a data structure mapping object types to symbols
    dsymbols = {'obstacle':'X', 'reward':'R'}

    # Make sure the object type is valid
    if object_type not in dsymbols.keys():
        return False
    
    # Find the positions of all of the empty cells in the grid.
    available_positions = find_available_positions(grid)
    
    # Make sure there are enough empty cells to place the objects.
    if len(available_positions) < n:
        print("not enough available spots")
        return False
    
    # Randomly choose *n* positions from available_positions and place an object at each of those positions.
    object_positions = random.sample(available_positions, n)
    for x, y in object_positions:
        grid[x][y] = dsymbols[object_type]
    return(True)


In [53]:
# Test your code
# Since the objects are added at random positions, we test the function 
# by counting the expected number of empty positions and object positions in the grid
test_grid = create_grid(4, 4)
print("New grid")
visualize_grid(test_grid)
add_objects(test_grid, object_type = "obstacle", n=4)
print("Obstacles added")
visualize_grid(test_grid)
print(sum([cell == '.' for cell in chain.from_iterable(test_grid)]) == 10) 
print(sum([cell == 'X' for cell in chain.from_iterable(test_grid)]) == 4)
print("Rewards added")
add_objects(test_grid, object_type = "reward", n=4)
visualize_grid(test_grid)
print(sum([cell == '.' for cell in chain.from_iterable(test_grid)]) == 6) 
print(sum([cell == 'X' for cell in chain.from_iterable(test_grid)]) == 4)
print(sum([cell == "R" for cell in chain.from_iterable(test_grid)]) == 4)



New grid
S . . .
. . . .
. . . .
. . . G
Obstacles added
S . . .
X X . .
. . X X
. . . G
True
True
Rewards added
S . . R
X X . .
R R X X
. R . G
True
True
True


# 2. Moving an agent in the grid world

We now need to implement methods to move an agent in different directions (north, south, east, or west) and update its position accordingly.

The first function determines the position where the agent will end up given a particular starting position and a particular action (north, south, east, or west).

This does not mean the agent will actually move to that position. It could be that the position is outside of the grid or that there is an obstacle at that position.

In [55]:
def intended_coordinates(position, action):
    """Determine the coordinates where the agent will end up if it takes a certain action.
    
    Arguments:
    - position (tuple): The current position of the agent.
    - action (str): The action to take (north, south, east, or west).

    Returns:
    - tuple: The coordinates where the agent wants to go it takes `action` at `position`. 
             This position can be off the grid (we will deal with that later).
    """
     # current position of the agent
    x, y = position    
    # find out where the agent wants to move
    if action == 'north': # row index (x) decreases
        intended_x = x - 1
        intended_y = y
    elif action == 'south': # row index (x) increases
        intended_x = x + 1
        intended_y = y
    elif action == 'west': # column index (y) decreases
        intended_x = x
        intended_y = y - 1
    elif action == 'east': # column index (y) increases
        intended_x = x
        intended_y = y + 1
    return(intended_x, intended_y)

In [56]:
# Test your code
# All tests should result in `True` if your code works correctly
print(intended_coordinates((0, 0), 'north') == (-1, 0))
print(intended_coordinates((3, 3), 'west') == (3, 2))
print(intended_coordinates((4, 4), 'east') == (4, 5))
print(intended_coordinates((3, 1), 'south') == (4, 1))

True
True
True
True


Next, you will write a function to determine whether a position is on-the grid or not.

In [57]:
def on_grid(grid, position):
    """Determine if a particular position is on the grid or not.
    
    Arguments:
    - grid (list): A 2D list representing the grid.
    - position (tuple): The position to check."""
    # bounds of the grid
    x_min=0
    y_min=0
    x_max=len(grid)-1
    y_max=len(grid[0])-1
    # current position of the agent
    x, y = position
    if x < x_min or x > x_max  or y < y_min or y > y_max:
        return False
    else:
        return True
    

In [ ]:
# Test your code
# All tests should result in `True` if your code works correctly
test_grid = create_grid(4, 4)
print(on_grid(test_grid, (0, 0)) == True)
print(on_grid(test_grid, (3, 3)) == True)
print(on_grid(test_grid, (4, 4)) == False)
print(on_grid(test_grid, (-1, 3)) == False)
print(on_grid(test_grid, (3, -1)) == False)

True
True
True
True
True


False

Now, you can write a functions to move the agent around in the frid. 

This function makes use of the `on_grid` and `intended_coordinates` functions.

In [60]:
def move_agent(grid, position, action, rewards={'goal':1, 'empty':-0.1, 'start': -0.1, 'obstacle':-0.2, 'off_grid':-0.2, 'reward':0.2}):
   
    """Move an agent in a grid world environment and collect the associated rewards.

    - If the intended coordinates are not on the grid, the agent remains in its original position and collects the value of `empty_reward`.
    - If the intended coordinates contain an obstacle (`X` symbol), the agent remains in its orginal position and collects the value of `obstacle_reward`.
    - In all other cases, the agent moves to the intended position and collects the associated reward.
    
    Arguments:
    - grid (list): A 2D list representing the grid world.
    - position (tuple): The current position of the agent.
    - action (str): The action to take (north, south, east, or west).
    - rewards (dict): A dictionary containing the rewards for each possible grid cell type.
    
    Returns:
    - position (tuple): The new position of the agent.
    - reward (int): The reward collected (if any).
    - terminated (bool): Whether the episode has ended.
    """
    # current position of the agent
    x, y = position    
    # find out where the agent wants to move
    intended_x, intended_y = intended_coordinates(position, action)
    # check if the agent is trying to move off-grid
    if not on_grid(grid, (intended_x, intended_y)):
        reward = rewards['off_grid']
        terminated = False
        return(position, reward, terminated)
    # if the agent is not trying to move off-grid, check if the new position is an obstacle
    elif grid[intended_x][intended_y] == 'X':
        reward = rewards['obstacle']
        terminated = False
        return(position, reward, terminated)
    # if the new position is not an obstacle or off-grid, check if the new position is a reward
    elif grid[intended_x][intended_y] == 'R':
        reward = rewards['reward']
        terminated = False
        # remove the reward from the grid
        grid[intended_x][intended_y] = '.'
        return((intended_x, intended_y), reward, terminated)
    # if the new position is not an obstacle or a reward, check if the new position is the goal
    elif grid[intended_x][intended_y] == 'G':
        reward = rewards['goal']
        terminated = True
        return((intended_x, intended_y), reward, terminated)
    # if the new position is not an obstacle, a reward, or the goal, check if the new position is an empty cell
    elif grid[intended_x][intended_y] == '.':
        reward = rewards['empty']
        terminated = False
        return((intended_x, intended_y), reward, terminated)
    # if the new position is not an obstacle, a reward, the goal, or an empty cell, check if the new position is the start
    elif grid[intended_x][intended_y] == 'S':
        reward = rewards['start']
        terminated = False
        return((intended_x, intended_y), reward, terminated)

Let's illustrate the function by moving an agent through the grid world in a predictable pattern. 
The agent will start by moving south until it reaches an obstacle or moves off-grid. Then it will turn 90 degrees counterclockwise and continue the same procedure.
The agent will stop after 20 moves or when it has reached the goal 

In [66]:
# Create a grid world
gw = create_grid(4,4)
# Add some obstacles
add_objects(gw, object_type = "obstacle", n=4)
# We don't need rewards for this example
# Define the possible actions as a cycle
actions = cycle(('south','east','north','west'))
# Initialize the agent 
max_moves = 20
terminated = False
position = (0,0)
action=next(actions)
cumulative_reward = 0
# Show the starting grid
visualize_grid(gw, position)
# Move the agent for `max_moves` cycles or until the goal is achieved
for _ in range(max_moves):
    print("\nMOVING: {:s}".format(action))
    new_position, reward, terminated = move_agent(gw, position, action)
    cumulative_reward+=reward
    print("REWARD: {:0.2f} TOTAL: {:0.2f}".format(reward, cumulative_reward))
    if new_position == position:
        action = next(actions)
    position = new_position
    visualize_grid(gw, position)
    if terminated:
        break


A . X .
. X . .
. . . X
. . X G

MOVING: south
REWARD: -0.10 TOTAL: -0.10
S . X .
A X . .
. . . X
. . X G

MOVING: south
REWARD: -0.10 TOTAL: -0.20
S . X .
. X . .
A . . X
. . X G

MOVING: south
REWARD: -0.10 TOTAL: -0.30
S . X .
. X . .
. . . X
A . X G

MOVING: south
REWARD: -0.20 TOTAL: -0.50
S . X .
. X . .
. . . X
A . X G

MOVING: east
REWARD: -0.10 TOTAL: -0.60
S . X .
. X . .
. . . X
. A X G

MOVING: east
REWARD: -0.20 TOTAL: -0.80
S . X .
. X . .
. . . X
. A X G

MOVING: north
REWARD: -0.10 TOTAL: -0.90
S . X .
. X . .
. A . X
. . X G

MOVING: north
REWARD: -0.20 TOTAL: -1.10
S . X .
. X . .
. A . X
. . X G

MOVING: west
REWARD: -0.10 TOTAL: -1.20
S . X .
. X . .
A . . X
. . X G

MOVING: west
REWARD: -0.20 TOTAL: -1.40
S . X .
. X . .
A . . X
. . X G

MOVING: south
REWARD: -0.10 TOTAL: -1.50
S . X .
. X . .
. . . X
A . X G

MOVING: south
REWARD: -0.20 TOTAL: -1.70
S . X .
. X . .
. . . X
A . X G

MOVING: east
REWARD: -0.10 TOTAL: -1.80
S . X .
. X . .
. . . X
. A X G

MOVING: ea

Repeat this procedure a few times and write down your observations about the behavior of the agent. How could you improve the behavior?

### My observations:

Now let's try moving the agent *randomly* through the grid world. 
The agent will start by moving in a random direction until it reaches an obstacle or moves off-grid. Then it will again take a random direction and continue the same procedure.
The agent will stop after 20 moves or when it has reached the goal.

In [75]:
# Create a grid world
gw = create_grid(4,4)
# Add some obstacles
add_objects(gw, object_type = "obstacle", n=4)
# We don't need rewards for this example
# Define the possible actions as a cycle
actions = ('south','east','north','west')
# Initialize the agent
max_moves = 20
terminated = False
position = (0,0)
action=random.choice(actions)
cumulative_reward = 0
# Show the starting grid
visualize_grid(gw, position)
# Move the agent for `max_moves` cycles or until the goal is achieved
for _ in range(max_moves):
    print("\nMOVING: {:s}".format(action))
    new_position, reward, terminated = move_agent(gw, position, action)
    cumulative_reward+=reward
    print("REWARD: {:0.2f} TOTAL: {:0.2f}".format(reward, cumulative_reward))
    if new_position == position:
        action = random.choice(actions)
    position = new_position
    visualize_grid(gw, position)
    if terminated:
        break


A . . X
X X . .
. . . X
. . . G

MOVING: west
REWARD: -0.20 TOTAL: -0.20
A . . X
X X . .
. . . X
. . . G

MOVING: south
REWARD: -0.20 TOTAL: -0.40
A . . X
X X . .
. . . X
. . . G

MOVING: north
REWARD: -0.20 TOTAL: -0.60
A . . X
X X . .
. . . X
. . . G

MOVING: west
REWARD: -0.20 TOTAL: -0.80
A . . X
X X . .
. . . X
. . . G

MOVING: south
REWARD: -0.20 TOTAL: -1.00
A . . X
X X . .
. . . X
. . . G

MOVING: north
REWARD: -0.20 TOTAL: -1.20
A . . X
X X . .
. . . X
. . . G

MOVING: east
REWARD: -0.10 TOTAL: -1.30
S A . X
X X . .
. . . X
. . . G

MOVING: east
REWARD: -0.10 TOTAL: -1.40
S . A X
X X . .
. . . X
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -1.60
S . A X
X X . .
. . . X
. . . G

MOVING: north
REWARD: -0.20 TOTAL: -1.80
S . A X
X X . .
. . . X
. . . G

MOVING: north
REWARD: -0.20 TOTAL: -2.00
S . A X
X X . .
. . . X
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -2.20
S . A X
X X . .
. . . X
. . . G

MOVING: south
REWARD: -0.10 TOTAL: -2.30
S . . X
X X A .
. . . X
. . . G

MOVING: sou

Repeat this procedure a few times and write down your observations about the behavior of the agent compared to the first procedure. 
How could you still improve the behavior?

### My observations:

# 3. Learning the Q-function
The above strategies were not very good. Every time the agent starts again, it will continue either making the same mistakes or making new mistakes.

Instead, we would like our agent to learn the best actions it can take in a particular state. Another way of saying this is that we want to approximate the *Q-function* (Q stands for quality).

This Q-function should reflect both the direct rewards the agent receives and the *total reward* the agent has collected when it reaches the goal state.

So how can we measure the success (or quality) of an action if the reward comes much later?

There are many ways to approach this problem, but one of the easiest is *direct utility estimation*.

Let's say that our agent has found a particular set of actions to reach the goal state. Then we can compute the rewards for each action the agent performs when it is in a particular state.

In [76]:
# Here is a pre-defined grid world
gw  =[['S', 'X', 'X', '.'], ['.', '.', '.', 'X'], ['.', '.', 'X', '.'], ['.', '.', '.', 'G']]

# It looks like this when the agent is at the start
# A X X .
# . . . X
# . . X .
# . . . G

# A set of actions the agent can take to reach the goal in this grid
actions = ['south','south','east','south','east','east']

# Let's keep track of the rewards the agent receives by taking these actions at each position
action_rewards = []
position = (0,0)
for action in actions:
    new_position, reward, terminated = move_agent(gw, position, action)
    action_rewards.append((position, action, reward))
    position = new_position
print(action_rewards)

[((0, 0), 'south', -0.1), ((1, 0), 'south', -0.1), ((2, 0), 'east', -0.1), ((2, 1), 'south', -0.1), ((3, 1), 'east', -0.1), ((3, 2), 'east', 1)]


## 3.1 Computing the reward-to-go

We can now compute the future rewards for each (state, action) pair by adding up the rewards for this step and all subsequent steps. This is also called the reward-to-go. 

There is also the possibility of using a *discount factor* (symbol: greek lowercase *gamma*) that is used to ensure that rewards that are further away count for less. We will add the discount factor to the formula, but we won't use it in the examples.

Implement the reward_to_go function using the following formula:

$R_t=\sum_{i=t}^{n} \gamma^i r_i$

This says that the reward-to-go at step *t* is the sum of all the rewards starting from time *t*, each reward being multiplied by a discount_factor raised to the power *t*.


In [79]:
def rewards_to_go(path, gamma=1.0):
    """
    Computes the reward-to-go for each step in a given path.
    
    Arguments:
    - path (list): A list of 3-tuples containing the position, action, and reward for each step in the path.
    - gamma (float): The discount factor.
    
    Returns:
    - forward_rewards (list): A list with 3-tuples consisting of position, action, and corresponding reward-to-go.
    """
    forward_rewards = []
    for t in range(len(path)):
        position, action, reward = path[t]
        rtg = sum([pow(gamma,t)*r for p,a,r in path[t:]])
        forward_rewards.append((position, action, rtg))
    return(forward_rewards)

Doing this with the actions from the previous example we can now compare the original rewards with the rewards-to-go. Write down your observations about this difference below.

In [81]:
print(action_rewards)
print(rewards_to_go(action_rewards))

[((0, 0), 'south', -0.1), ((1, 0), 'south', -0.1), ((2, 0), 'east', -0.1), ((2, 1), 'south', -0.1), ((3, 1), 'east', -0.1), ((3, 2), 'east', 1)]
[((0, 0), 'south', 0.5), ((1, 0), 'south', 0.6), ((2, 0), 'east', 0.7), ((2, 1), 'south', 0.8), ((3, 1), 'east', 0.9), ((3, 2), 'east', 1.0)]


My observations:

## 3.2 Wandering to collect information
If we would have different rewards-to-go for different actions when the agent is in a particular position, we could choose the action with the highest reward-to-go.

But how do we achieve this?

To figure out the expected reward-to-go for each action, we can let the agent wander around the environment to collect information. 

We will let the agent wander until it has reached the goal state or until it has exceed a pre-defined number of steps. We call this an *episode*.

In each episode, the agent will collect information about the path it has taken and the rewards collected. 

Below you will find the `wander` function.

In [82]:
def wander(grid, max_steps=20):
    """Let an agent wander around a grid until max_steps are reached or the goal is achieved.

    After the episode has ended, the (position, action, reward) tuples are returned.
    
    Arguments:
    - grid (list): A 2D list representing the grid world.
    - max_steps (int): The maximum number of steps to take before terminating.
    - agent_position (tuple): The starting position of the agent.

    Returns:
    - path (list): A list of positions taken by the agent.
    - rewards (list): A list of rewards collected by the agent.
    """
    # Make a copy of the grid so we don't modify the original
    grid = copy.deepcopy(grid)
    actions = ('north', 'south', 'west', 'east')
    terminated = False
    position = (0,0)
    action_rewards = []
    # Move the agent randomly until the max number of steps has been exceeded or until goal is achieved
    for step in range(max_steps):
        action=random.choice(actions)
        new_position, reward, terminated = move_agent(grid, position, action)
        action_rewards.append((position, action, reward))
        position = new_position
        if terminated:
            break
    return(action_rewards)

Run the code below to let the agent wander around a few times and write down your observations on the behavior of the agent and the information collected about the rewards-to-go.

In [83]:
# Here is that pre-defined grid world again
gw  =[['S', 'X', 'X', '.'], ['.', '.', '.', 'X'], ['.', '.', 'X', '.'], ['.', '.', '.', 'G']]
visualize_grid(gw)

S X X .
. . . X
. . X .
. . . G


In [86]:
# Let the agent wander in the grid world
action_rewards = wander(gw)
rewards_to_go(action_rewards)

[((0, 0), 'west', -2.6),
 ((0, 0), 'north', -2.4000000000000004),
 ((0, 0), 'west', -2.2),
 ((0, 0), 'south', -2.0),
 ((1, 0), 'north', -1.9000000000000001),
 ((0, 0), 'north', -1.8),
 ((0, 0), 'north', -1.6),
 ((0, 0), 'south', -1.4000000000000001),
 ((1, 0), 'south', -1.3),
 ((2, 0), 'east', -1.2000000000000002),
 ((2, 1), 'north', -1.1),
 ((1, 1), 'south', -1.0),
 ((2, 1), 'north', -0.9),
 ((1, 1), 'north', -0.8),
 ((1, 1), 'east', -0.6000000000000001),
 ((1, 2), 'west', -0.5),
 ((1, 1), 'east', -0.4),
 ((1, 2), 'west', -0.30000000000000004),
 ((1, 1), 'west', -0.2),
 ((1, 0), 'north', -0.1)]

My observations:


## 3.3 Computing the expected reward-to-go

Now we'll let the agent wander for many episodes and we will compute the average reward (or expected total future reward for that state) based on the results of all of the episodes. We can also call this the *Q-table*.

In [87]:
def expected_rewards_to_go(episodes):
    """Compute the expected reward-to-go for each position and action, given a list of episodes.

    Arguments:
    - episodes (list): A list of episodes, where each episode is a list of 3-tuples containing the position, action, and reward for each step in the episode.
    
    Returns:
    - expected_rewards (list): A dictionary with position as key and a subdictionary containing the expected reward-to-go for each action.
    """
    # Go through all the rewards-to-go for each position and action in each episode.
    # By storing them in a dictionary with (position, action) as the key and a list of (action,reward) tuples as the value.
    # we can easily compute the averages later
    rtg_data = defaultdict(list)
    for episode in episodes:
        for position, action, reward in rewards_to_go(episode):
            rtg_data[(position, action)].append(reward)
    # Compute the expected reward-to-go for each position and action
    expected_rewards = defaultdict(list)
    for (position, action), rewards in rtg_data.items():
        expected_rewards[position].append((action, sum(rewards)/len(rewards)))
    return(expected_rewards)

In [89]:
# We'll use the same pre-defined grid world
gw  =[['S', 'X', 'X', '.'], ['.', '.', '.', 'X'], ['.', '.', 'X', '.'], ['.', '.', '.', 'G']]
episode_data = []
n_episodes = 100
for _ in range(n_episodes):
    episode_data.append(wander(gw))
q_table = expected_rewards_to_go(episode_data)


In [90]:
q_table

defaultdict(list,
            {(0, 0): [('north', -2.083116883116883),
              ('west', -2.1531914893617023),
              ('south', -1.86319018404908),
              ('east', -2.0981250000000005)],
             (1, 0): [('west', -1.6103773584905663),
              ('south', -1.3511627906976744),
              ('east', -1.5303370786516857),
              ('north', -1.5986666666666667)],
             (2, 0): [('south', -1.1408163265306124),
              ('west', -1.1318181818181818),
              ('east', -0.9746268656716419),
              ('north', -1.0121951219512195)],
             (3, 0): [('west', -1.0346153846153847),
              ('north', -0.9823529411764708),
              ('east', -0.8212121212121213),
              ('south', -1.2869565217391306)],
             (2, 1): [('south', -0.8630434782608696),
              ('west', -0.9148936170212766),
              ('east', -0.8487179487179487),
              ('north', -0.9263157894736843)],
             (3, 1): [('south'

## 3.4 Finding the best action

Now that we have learned a bit more about the expected rewards for different actions in different states, we can let the agent select the action with the highest utility in each state. 

Let's write a function that extracts the best action in a particular position from a Q-table. 

In [91]:
def best_action(q_table, position):
    """Find the best action for a given position, given a Q-table.

    Arguments:
    - q_table (dict): A dictionary of the expected reward-to-go for each position and action.
    - position (tuple): The position to find the best action for.

    Returns:
    - best_action (str): The best action for the given position.
    """
    # Find the action with the highest expected reward-to-go for the given position
    available_actions = q_table[position]
    return(max(available_actions, key=lambda x: x[1])[0])

In [93]:
# Again, use the same pre-defined grid world
gw  =[['S', 'X', 'X', '.'], ['.', '.', '.', 'X'], ['.', '.', 'X', '.'], ['.', '.', '.', 'G']]

# Initialize the agent
terminated = False
position = (0,0)
reward = 0
cumulative_reward = 0
# Show the starting grid
visualize_grid(gw, position)
# For each position, perform the action with the highest expected reward-to-go in the q-table
for _ in range(max_moves):
    action = best_action(q_table, position)
    print("\nMOVING: {:s}".format(action))
    new_position, reward, terminated = move_agent(gw, position, action)
    cumulative_reward+=reward
    print("REWARD: {:0.2f} TOTAL: {:0.2f}".format(reward, cumulative_reward))
    position = new_position
    visualize_grid(gw, position)
    if terminated:
        break

A X X .
. . . X
. . X .
. . . G

MOVING: south
REWARD: -0.10 TOTAL: -0.10
S X X .
A . . X
. . X .
. . . G

MOVING: south
REWARD: -0.10 TOTAL: -0.20
S X X .
. . . X
A . X .
. . . G

MOVING: east
REWARD: -0.10 TOTAL: -0.30
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -0.50
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -0.70
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -0.90
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -1.10
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -1.30
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -1.50
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -1.70
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -1.90
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -2.10
S X X .
. . . X
. A X .
. . . G

MOVING: east
REWARD: -0.20 TOTAL: -2.30
S X X .
. . . X
. A X .
. . . G

MOVING: east
REW

Write down your observations. How does the agent behave in this case?

### My observations:

## 4. Putting everything together
Below, we will define some wrapper functions to do simulations more easily. 

In [95]:
def derive_qtable(grid, n_episodes):
    """Given a grid world and a number of episodes, derive the Q-table.
    
    Arguments:
    - grid (list): A 2D list representing the grid world.
    - n_episodes (int): The number of episodes to use for deriving the Q-table.
    
    Returns:
    - q_table (dict): A dictionary of the expected reward-to-go for each position and action.
    """
    episode_data = []
    for _ in range(n_episodes):
        episode_data.append(wander(grid))
    q_table = expected_rewards_to_go(episode_data)
    return(q_table)


In [96]:
def navigate(grid, q_table):
    """Let an agent navigate a grid world using the Q-table.
    
    Arguments:
    - grid (list): A 2D list representing the grid world.
    - q_table (dict): A dictionary of the expected reward-to-go for each position and action.
    """
    # Initialize the agent
    terminated = False
    position = (0,0)
    reward = 0
    cumulative_reward = 0
    # Show the starting grid
    visualize_grid(grid, position)
    # For each position, perform the action with the highest expected reward-to-go in the q-table
    for _ in range(max_moves):
        action = best_action(q_table, position)
        print("\nMOVING: {:s}".format(action))
        new_position, reward, terminated = move_agent(grid, position, action)
        cumulative_reward+=reward
        print("REWARD: {:0.2f} TOTAL: {:0.2f}".format(reward, cumulative_reward))
        position = new_position
        visualize_grid(grid, position)
        if terminated:
            break

Now let's create a larger grid world with additional rewards (see the `move_agent` function to understand the value of these rewards). 

In [98]:
# larger grid world with additional rewards
lgw = create_grid(5, 5)
add_objects(lgw, object_type="obstacle", n=5)
add_objects(lgw, object_type="reward", n=5)
# derive the Q-table using 100 episodes
q_table = derive_qtable(lgw, 10000)
# show how the agent navigates the grid world given the Q-table
navigate(lgw, q_table)

A . R X .
R . . X .
R X . . .
. R . R .
X X . . G

MOVING: south
REWARD: 0.20 TOTAL: 0.20
S . R X .
A . . X .
R X . . .
. R . R .
X X . . G

MOVING: south
REWARD: 0.20 TOTAL: 0.40
S . R X .
. . . X .
A X . . .
. R . R .
X X . . G

MOVING: south
REWARD: -0.10 TOTAL: 0.30
S . R X .
. . . X .
. X . . .
A R . R .
X X . . G

MOVING: east
REWARD: 0.20 TOTAL: 0.50
S . R X .
. . . X .
. X . . .
. A . R .
X X . . G

MOVING: east
REWARD: -0.10 TOTAL: 0.40
S . R X .
. . . X .
. X . . .
. . A R .
X X . . G

MOVING: east
REWARD: 0.20 TOTAL: 0.60
S . R X .
. . . X .
. X . . .
. . . A .
X X . . G

MOVING: south
REWARD: -0.10 TOTAL: 0.50
S . R X .
. . . X .
. X . . .
. . . . .
X X . A G

MOVING: east
REWARD: 1.00 TOTAL: 1.50
S . R X .
. . . X .
. X . . .
. . . . .
X X . . A


Experiment with the grid size and the number of obstacles and rewards. Write down your observations. 

### My observations: